In [ ]:
import os
print(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

In [ ]:
!ls -R /kaggle

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import sklearn
df=pd.read_csv("/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
df.sample()

**BERT stands for Bidirectional Encoder Representations from Transformers and it is a state-of-the-art machine learning model used for NLP tasks. Jacob Devlin and his colleagues developed BERT at Google in 2018. Devlin and his colleagues trained the BERT on English Wikipedia (2,500M words) and BooksCorpus (800M words) and achieved the best accuracies for some of the NLP tasks in 2018. There are two pre-trained general BERT variations: The base model is a 12-layer, 768-hidden, 12-heads, 110M parameter neural network architecture, whereas the large model is a 24-layer, 1024-hidden, 16-heads, 340M parameter neural network architecture.**

# Sentiment Analysis with BERT
We will do the following operations to train a sentiment analysis model:
* Install Transformers library;
* Load the BERT Classifier and Tokenizer alıng with Input modules;
* Download the IMDB Reviews Data and create a processed dataset (this will take several operations;
* Configure the Loaded BERT model and Train for Fine-tuning
* Make Predictions with the Fine-tuned Model

In [ ]:
# Installing Transformers library
# !pip install transformers 

In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import InputExample
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
# model.summary()

In [ ]:
# changing positive and negative into numeric values

def cat2num(value):
    if value=='positive': 
        return 1
    else: 
        return 0
    
df['sentiment']  =  df['sentiment'].apply(cat2num)
train = df[:45000]
# train = df[:4000]
test = df[45000:]

# Data Preprocessing
For training model with BERT, we need to do some additional Prepriocessing. Let's understand them one by one!
* Add special tokens to separate sentences and do classification
* Pass sequences of constant length (introduce padding)
* Create array of 0s (pad token) and 1s (real token) called attention mask

In [ ]:
# But first see BERT tokenizer exmaples and other required stuff!

example='In this Kaggle notebook, I will do sentiment analysis using BERT with Huggingface'
tokens=tokenizer.tokenize(example)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(tokens)
print(token_ids)

- > The original word has been split into smaller subwords and characters. This is because Bert Vocabulary is fixed with a size of ~30K tokens. Words that are not part of vocabulary are represented as subwords and characters.

- > Tokenizer takes the input sentence and will decide to keep every word as a whole word, split it into sub words(with special representation of first sub-word and subsequent subwords — see ## symbol in the example above) or as a last resort decompose the word into individual characters. Because of this, we can always represent a word as, at the very least, the collection of its individual characters.

Reference: https://medium.com/@dhartidhami/understanding-bert-word-embeddings-7dc4d2ea54ca

### Special Tokens
* [SEP] - marker for ending of a sentence
* [CLS] - we must add this token to the start of each sentence, so BERT knows we’re doing classification
* [PAD] -There is also a special token for padding:
* [UNK] - ERT understands tokens that were in the training set. Everything else can be encoded using the [UNK] (unknown) token

1. — ***convert_data_to_examples***: This will accept our train and test datasets and convert each row into an InputExample object.
2. — ***ReviewDataset***: This function will tokenize the InputExample objects, then create the required input format with the tokenized objects, finally, create an input dataset that we can feed to the model.


In [ ]:
def convert_data_to_examples(train, test, review, sentiment): 
    train_InputExamples = train.apply(lambda x: InputExample(guid=None, # Globally unique ID for bookkeeping, unused in this case
                                                          text_a = x[review], 
                                                          label = x[sentiment]), axis = 1)

    validation_InputExamples = test.apply(lambda x: InputExample(guid=None, # Globally unique ID for bookkeeping, unused in this case
                                                          text_a = x[review], 
                                                          label = x[sentiment]), axis = 1,)
  
    return train_InputExamples, validation_InputExamples

train_InputExamples, validation_InputExamples = convert_data_to_examples(train,  test, 'review',  'sentiment')
                                                                         

In [ ]:
# train_InputExamples[0]

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=128):
        self.features = []

        for e in tqdm(examples):
            encoded = tokenizer(
                e.text_a,
                add_special_tokens=True,         # [CLS], [SEP]
                max_length=max_length,
                padding='max_length',            # pad to max_length
                truncation=True,
                return_attention_mask=True,
                return_token_type_ids=True,
            )

            self.features.append({
                "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
                "token_type_ids": torch.tensor(encoded["token_type_ids"], dtype=torch.long),
                "labels": torch.tensor(e.label, dtype=torch.long)
            })

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]

In [ ]:
train_dataset = ReviewDataset(train_InputExamples, tokenizer, max_length=128)
val_dataset = ReviewDataset(validation_InputExamples, tokenizer, max_length=128)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [ ]:
## Our dataset containing processed input sequences are ready to be fed to the model.

In [ ]:
import torch.nn as nn
from transformers import DistilBertModel

class DistilBERTMeanPoolClassifier(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_labels=2, dropout=0.2):
        super().__init__()
        
        # DistilBERT backbone
        self.backbone = DistilBertModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size  # usually 768
        
        # 2-layer classifier head
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels)
        )

    def mean_pool(self, last_hidden_state, attention_mask):
        """
        last_hidden_state: (B, T, H)
        attention_mask:    (B, T)
        """
        mask = attention_mask.unsqueeze(-1).float()   # (B, T, 1)
        masked_hidden = last_hidden_state * mask      # pad tokens become 0
        summed = masked_hidden.sum(dim=1)             # (B, H)
        counts = mask.sum(dim=1).clamp(min=1e-9)      # (B, 1)
        mean_pooled = summed / counts                 # (B, H)
        return mean_pooled

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # token-level outputs: (B, T, H)
        last_hidden_state = outputs.last_hidden_state
        
        # mean pooling
        pooled = self.mean_pool(last_hidden_state, attention_mask)
        
        # classifier head
        logits = self.classifier(pooled)
        return logits
    

model = DistilBERTMeanPoolClassifier(
    model_name="distilbert-base-uncased",
    num_labels=2,
    dropout=0.2
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DistilBERTMeanPoolClassifier(
    model_name="distilbert-base-uncased",
    num_labels=2,
    dropout=0.2
).to(device)

criterion = nn.CrossEntropyLoss()

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
# Freeze DistilBERT backbone
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

stage1_epochs = 1

for epoch in range(stage1_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    print(f"[Stage 1] Epoch {epoch+1}/{stage1_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 50)

In [ ]:
# Unfreeze all parameters
for param in model.backbone.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

stage2_epochs = 2

for epoch in range(stage2_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    print(f"[Stage 2] Epoch {epoch+1}/{stage2_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 50)

In [ ]:
# for param in model.backbone.parameters():
#     param.requires_grad = False
# optimizer = torch.optim.AdamW(
#     filter(lambda p: p.requires_grad, model.parameters()),
#     lr=1e-3
# )


# optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
# criterion = nn.CrossEntropyLoss()
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# model.train()

# for batch in train_loader:
#     input_ids = batch["input_ids"].to(device)
#     attention_mask = batch["attention_mask"].to(device)
#     labels = batch["labels"].to(device)

#     optimizer.zero_grad()

#     logits = model(
#         input_ids=input_ids,
#         attention_mask=attention_mask
#     )

#     loss = criterion(logits, labels)
#     loss.backward()
#     optimizer.step()

In [ ]:
# model.eval()

# correct = 0
# total = 0

# with torch.no_grad():
#     for batch in val_loader:
        
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         labels = batch["labels"].to(device)

#         logits = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask
#         )

#         preds = torch.argmax(logits, dim=1)

#         correct += (preds == labels).sum().item()
#         total += labels.size(0)

# accuracy = correct / total

# print("Validation Accuracy:", accuracy)